In [1]:
import pandas as pd
from tqdm import tqdm

In [2]:
results = pd.read_csv("../results/real_results/diffnaps/synthetic.tsv", sep="\t")

In [3]:
data = pd.read_csv("../../data/synthetic3_all.csv")

In [4]:
results.columns = results.columns.str.strip()

In [5]:
target_tuple = ("Target", 1)
target_entry = data[target_tuple[0]] == target_tuple[1]
global_pattern_entry = pd.Series(False, index = data.index)

In [6]:
max_jaccard = 0
previous_entries = []
n_subgroups = 0
sizes = []
for i,r in results.iterrows():
    if r["Label"] != target_tuple[1]:
        continue
    pattern = eval(r["Pattern"])
    pattern_entry = pd.Series(True, index = data.index)
    for p in pattern:
        pattern_entry &= (data[p] == 1)
    n_subgroups += 1
    sizes.append(len(pattern))
    for e in previous_entries:
        try:
            jaccard = sum(pattern_entry & e) / sum(pattern_entry | e)
        except ZeroDivisionError:
            jaccard = 0
        max_jaccard = max(max_jaccard, jaccard)
    previous_entries.append(pattern_entry)
    global_pattern_entry |= pattern_entry

In [7]:
print("Model:", "DiffNAPS")
print("Dataset:", "Synthetic")
print("# Subgroups:", n_subgroups)
print("Avg. size:", sum(sizes) / n_subgroups)
print("Maximum Jaccard:", max_jaccard)

Model: DiffNAPS
Dataset: Synthetic
# Subgroups: 1
Avg. size: 1.0
Maximum Jaccard: 0


In [8]:
# global_pattern_entry.value_counts()
positive = (global_pattern_entry & target_entry).sum() / target_entry.sum()
negative = (global_pattern_entry & ~target_entry).sum() / (~target_entry).sum()
youden = positive- negative
youden


0.25